# CareCompanion: Skills, Memory & Dreams for Health Agents
## Powered by Gemma 4 — Local GPU Inference

**[Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon) — Health & Sciences Track**

This notebook demonstrates CareCompanion running **Gemma 4 locally on GPU** using HuggingFace Transformers. The model runs entirely on your GPU — no API keys, no cloud calls, full privacy.

### Three Agent Primitives

1. **Skills** — Reusable expertise modules loaded on-demand (progressive disclosure)
2. **Memory** — Persistent, SHA256-versioned health records across sessions
3. **Dreams** — Offline memory consolidation that discovers clinical insights

### GPU Requirements

| Model | VRAM | Colab Tier | Notes |
|-------|------|------------|-------|
| **E4B-it** (default) | ~8 GB | Free (T4) | Best quality/speed for free tier |
| E2B-it | ~4 GB | Free (T4) | Smallest, fastest |
| 26B-A4B-it (4-bit) | ~14 GB | Free (T4) | MoE, best quality, needs quantization |
| 26B-A4B-it (bf16) | ~52 GB | Pro (A100) | Full precision |

---

**GitHub**: [EvolvingAgentsLabs/skillos_x_mobile](https://github.com/EvolvingAgentsLabs/skillos_x_mobile)  
**Video**: [YouTube Demo](https://youtu.be/8ho3SpY-rDU)

> This notebook runs Gemma 4 **entirely on your GPU** — demonstrating that open-weight models can power healthcare agents with full data privacy, no internet required after model download.

---
## Setup

**Requirements:**
1. **GPU runtime** — Go to Runtime → Change runtime type → select **T4 GPU**
2. **HuggingFace access** — Accept the [Gemma 4 license](https://huggingface.co/google/gemma-4-E4B-it) and add your HF token as a Colab Secret named `HF_TOKEN`

No API keys needed — the model runs locally on your GPU.

In [1]:
# Install dependencies for local GPU inference
# Gemma 4 requires the latest transformers from GitHub (not yet in PyPI release)
!pip install -q torch accelerate bitsandbytes
!pip install -q git+https://github.com/huggingface/transformers.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 52.6 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Clone the CareCompanion repository (skills, memory, traces data)
import os
PROJECT_DIR = '/content/skillos_x_mobile'
if not os.path.exists(PROJECT_DIR):
    !git clone -q https://github.com/EvolvingAgentsLabs/skillos_x_mobile.git {PROJECT_DIR}
    print('Repository cloned.')
else:
    print('Repository already present.')

Repository cloned.


In [3]:
# ── Select model and load on GPU ──

MODEL_ID = 'google/gemma-4-26B-A4B-it'  # @param ['google/gemma-4-E2B-it', 'google/gemma-4-E4B-it', 'google/gemma-4-26B-A4B-it']
USE_4BIT = False  # @param {type:"boolean"}

import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

# HuggingFace authentication
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU')

# Auto-enable 4-bit for 26B on GPUs with <40GB
is_large_model = '26B' in MODEL_ID or '31B' in MODEL_ID
if is_large_model and gpu_mem < 40 and not USE_4BIT:
    print(f'Auto-enabling 4-bit quantization for {MODEL_ID} on {gpu_name} ({gpu_mem:.0f}GB)')
    USE_4BIT = True

# Load model
print(f'Loading {MODEL_ID}...')
load_kwargs = {'device_map': 'auto'}

if USE_4BIT:
    load_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type='nf4',
    )
else:
    load_kwargs['torch_dtype'] = 'auto'

model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, **load_kwargs)
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Memory usage
mem_used = torch.cuda.memory_allocated() / 1e9
print(f'Model loaded: {mem_used:.1f} GB VRAM used')
print(f'Quantized: {USE_4BIT}')

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (102.0 GB)
Loading google/gemma-4-26B-A4B-it...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Model loaded: 51.6 GB VRAM used
Quantized: False


---
## 1. Test Local Gemma 4

Quick test to verify the model is loaded and generating correctly on your GPU.

In [4]:
# Quick test — verify the model generates correctly
test_messages = [{'role': 'user', 'content': 'Say "CareCompanion ready" in exactly 2 words.'}]
test_text = processor.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
test_inputs = processor(text=test_text, return_tensors='pt').to(model.device)
test_out = model.generate(**test_inputs, max_new_tokens=10)
test_response = processor.decode(test_out[0][len(test_inputs['input_ids'][0]):], skip_special_tokens=True)
print(f'Test: {test_response.strip()}')
print(f'Model: {MODEL_ID}')
print(f'Device: {model.device}')

Test: CareCompanion ready
Model: google/gemma-4-26B-A4B-it
Device: cuda:0


---
## 2. Skills — Progressive Disclosure

Skills are reusable expertise modules stored as markdown files with YAML frontmatter. The agent only sees skill **names and descriptions** in its system prompt (~100 tokens/skill). When it needs a skill, it calls `load_skill(name)` to retrieve the full instructions.

This is functionally equivalent to [Anthropic's Skills API](https://docs.anthropic.com/en/docs/agents-and-tools/managed-agents#skills) — but implemented with open-weight Gemma 4 running locally.

In [5]:
import glob, re, json
from IPython.display import display, Markdown

# Load all skills
skill_files = sorted(glob.glob(f'{PROJECT_DIR}/skills/*.skill.md'))
skills = []

table = '| Skill | Description | Size |\n|-------|-------------|------|\n'
for sf in skill_files:
    with open(sf) as f:
        content = f.read()
    fm = re.match(r'^---\s*\n([\s\S]*?)\n---\s*\n([\s\S]*)$', content)
    if not fm:
        continue
    name = re.search(r'^name:\s*(.+)', fm.group(1), re.M)
    desc = re.search(r'^description:\s*(.+)', fm.group(1), re.M)
    name = name.group(1).strip() if name else os.path.basename(sf)
    desc = desc.group(1).strip() if desc else ''
    instructions = fm.group(2).strip()
    skills.append({'name': name, 'description': desc, 'instructions': instructions})
    table += f'| `{name}` | {desc} | ~{len(instructions.split())} words |\n'

display(Markdown(f'### {len(skills)} Skills Available\n\n' + table))
display(Markdown(
    '> **Level 1** (system prompt): Only name + description above (~400 tokens total).  \n'
    '> **Level 2** (on demand): Full instructions loaded only when `load_skill()` is called.'
))

### 4 Skills Available

| Skill | Description | Size |
|-------|-------------|------|
| `daily-routine` | Orchestrate the full daily wellness routine — check schedule, sequence care activities, and provide a daily summary. | ~247 words |
| `exercise-coaching` | Guide the user through a personalized exercise routine with voice coaching and progress tracking. | ~329 words |
| `medication-reminder` | Check medication schedule, remind the user to take their meds, confirm adherence, and log results. | ~214 words |
| `social-checkin` | Check on the user's emotional wellbeing, facilitate family connections, and log mood observations. | ~287 words |


> **Level 1** (system prompt): Only name + description above (~400 tokens total).  
> **Level 2** (on demand): Full instructions loaded only when `load_skill()` is called.

In [6]:
# Show one skill in detail
display(Markdown('### Skill Detail: `medication-reminder`\n\n'
                 'This is what the agent receives when it calls `load_skill("medication-reminder")`:\n'))
med_skill = next(s for s in skills if s['name'] == 'medication-reminder')
display(Markdown(med_skill['instructions']))

### Skill Detail: `medication-reminder`

This is what the agent receives when it calls `load_skill("medication-reminder")`:


## Instructions

### Overview
Guide the user through their medication routine, ensuring each medication is taken on time and properly logged.

### Procedure

1. **Check time**: Call `get_current_time()` to know when this session is happening.
2. **Read health profile**: Call `read_memory("health-profile", "medications.md")` to get the medication schedule.
3. **Check daily log**: Call `read_memory("daily-log", "today.md")` to see what's already been done today.
4. **Identify due medications**: Compare current time against the medication schedule.
5. **For each due medication**:
   a. `speak()`: "It's time for your [medication name] — [dosage]."
   b. `listen()` for the user's response.
   c. If they confirm taking it, mark as done.
   d. If they express concern or side effects, note them.
   e. If they skip, ask why and log the reason.
6. **Set reminder**: If medications are upcoming, call `set_alarm(time, label)` for the next one.
7. **Send notification**: Call `send_notification()` with a summary of what was taken.
8. **Log results**: Call `write_memory("daily-log", "today.md", ...)` with medication status.

### Safety Rules

- Never give medical advice — only remind about prescribed medications.
- If the user reports a new symptom or side effect, acknowledge it and suggest contacting their doctor.
- If a medication was already taken today (per daily log), do not remind again.
- Always confirm before marking a medication as taken.

---
## 3. Memory — Persistent Health Records

Memory stores are directories of versioned markdown documents. Every write creates a SHA256-versioned backup. This is functionally equivalent to [Anthropic's Memory API](https://docs.anthropic.com/en/docs/agents-and-tools/managed-agents#memory).

Our patient is **Maria**, a 72-year-old managing diabetes, hypertension, and knee arthritis.

In [7]:
import hashlib

class MemoryStore:
    """Simple Python memory store matching the TypeScript implementation."""
    def __init__(self, store_dir):
        self.store_dir = store_dir
        with open(os.path.join(store_dir, '_manifest.json')) as f:
            self.manifest = json.load(f)
        self.name = self.manifest['name']
    
    def read(self, doc_path):
        full = os.path.join(self.store_dir, doc_path)
        if not os.path.exists(full):
            return None
        with open(full) as f:
            content = f.read()
        sha = hashlib.sha256(content.encode()).hexdigest()
        return {'content': content, 'sha256': sha, 'version': 1}
    
    def write(self, doc_path, content):
        full = os.path.join(self.store_dir, doc_path)
        os.makedirs(os.path.dirname(full), exist_ok=True)
        with open(full, 'w') as f:
            f.write(content)
        sha = hashlib.sha256(content.encode()).hexdigest()
        return {'ok': True, 'sha256': sha}
    
    def list_docs(self):
        docs = []
        for root, dirs, files in os.walk(self.store_dir):
            for fn in files:
                if fn.startswith('_') or fn.startswith('.') or re.search(r'\.v\d+$', fn):
                    continue
                rel = os.path.relpath(os.path.join(root, fn), self.store_dir)
                docs.append(rel)
        return sorted(docs)

# Load all memory stores
memory_dir = f'{PROJECT_DIR}/memory'
memory_stores = {}
for name in os.listdir(memory_dir):
    manifest = os.path.join(memory_dir, name, '_manifest.json')
    if os.path.exists(manifest):
        memory_stores[name] = MemoryStore(os.path.join(memory_dir, name))

# Display stores
table = '| Store | Access | Documents | Description |\n|-------|--------|-----------|-------------|\n'
for s in sorted(memory_stores.values(), key=lambda x: x.name):
    docs = s.list_docs()
    table += f'| `{s.name}` | {s.manifest["access"]} | {len(docs)} | {s.manifest["description"]} |\n'
display(Markdown(f'### Memory Stores ({len(memory_stores)})\n\n' + table))

### Memory Stores (3)

| Store | Access | Documents | Description |
|-------|--------|-----------|-------------|
| `consolidated` | read_write | 4 | Dream-consolidated care memories. Health patterns, medication adherence, exercise progress, and social interaction history. |
| `daily-log` | read_write | 2 | Daily activity logs: medication adherence, exercise completion, mood observations, and social interactions. |
| `health-profile` | read_write | 3 | User health information: medications, exercise preferences, mobility level, family contacts, and care notes. |


In [8]:
# Show the patient's health profile
display(Markdown('### Patient Health Profile — Maria, 72 years old\n'))

hp = memory_stores.get('health-profile')
if hp:
    for doc_path in hp.list_docs():
        doc = hp.read(doc_path)
        if doc:
            display(Markdown(f'---\n#### `health-profile/{doc_path}`\n\n{doc["content"]}'))

### Patient Health Profile — Maria, 72 years old


---
#### `health-profile/exercise.md`

# Exercise Profile

## Mobility Level
- Seated exercises preferred — uses wheelchair most of day
- Knee osteoarthritis (bilateral, moderate) — avoid high-impact or weight-bearing
- Good upper body strength and range of motion
- Can stand with walker for 2-3 minutes at a time
- Balance: fair when supported, poor unsupported

## Preferred Exercises
- Neck rolls and shoulder work (enjoys these, finds them relaxing)
- Arm circles and overhead stretches
- Seated marching (when knees feel OK, usually mornings)
- Deep breathing and progressive relaxation (favorite cool-down)
- Gentle torso twists (helps with back stiffness)

## Exercises to Avoid
- Leg extensions (causes sharp knee pain)
- Standing exercises longer than 2 minutes
- Rapid or bouncing movements
- Weighted exercises (wrist strain from arthritis)

## Schedule
- Preferred time: 10:00 AM (after morning medications settle)
- Duration: 15-20 minutes
- Frequency: Daily target, averaging 5 of 7 days
- Best days: Monday-Wednesday (energy higher early in week)
- Challenging days: Thursday-Friday (tends to feel more fatigued)

## Recent Progress
- Week of May 5: completed 4/7 sessions, skipped Thursday (knee flare) and Saturday-Sunday
- Week of May 12: completed 3/5 sessions so far, increasing arm circle reps from 8 to 10
- Pain level during exercise: typically 2-3/10, manageable
- Reports feeling "looser and more awake" after sessions


---
#### `health-profile/medications.md`

# Medications

## Current Prescriptions

| Medication | Dosage | Schedule | Purpose | Notes |
|-----------|--------|----------|---------|-------|
| Lisinopril | 10mg | Morning (8:00 AM) | Blood pressure | Take with water, before breakfast |
| Metformin | 500mg | Morning (8:00 AM) + Evening (6:00 PM) | Type 2 diabetes | Take with food to reduce nausea |
| Vitamin D3 | 2000 IU | Morning (8:00 AM) | Bone health / deficiency | Doctor confirmed dose on last visit |
| Aspirin (low-dose) | 81mg | Evening (6:00 PM) | Cardiovascular prevention | Take with dinner |

## Adherence Notes
- Morning medications usually taken on time
- Evening Metformin sometimes forgotten (3 of last 7 days missed)
- Vitamin D occasionally skipped when running low on supply

## Last Refill
- Lisinopril: refilled May 1, 2026 (30-day supply)
- Metformin: refilled April 28, 2026 (30-day supply)
- Vitamin D3: running low — needs refill soon
- Aspirin: OTC, purchased May 5

## Doctor Notes
- Next appointment: May 22, 2026 with Dr. Ramirez
- Blood pressure trending well (last reading: 128/82)
- HbA1c at 6.8% — target is <7.0%, slight improvement from 7.1%
- Vitamin D levels were low at 22 ng/mL (target >30)


---
#### `health-profile/social.md`

# Social Connections

## Family

### Daughter — Sofia
- Lives in Buenos Aires (different city)
- Calls daily around 12:00 PM, video call preferred
- Works as a teacher, sometimes calls late if busy with classes
- Has two children: Lucas (8) and Valentina (5) — user loves hearing about them
- Topic favorites: grandchildren's school, Sofia's cooking experiments, family news

### Son — Miguel
- Lives locally, visits on weekends (Saturday or Sunday)
- Works as an engineer, busy during weekdays
- Brings groceries and helps with errands during visits
- Less talkative than Sofia but very attentive
- User appreciates his practical help

## Social Patterns
- Morning: low social need, focused on health routine
- Midday (11 AM - 1 PM): peak social energy, best time for calls
- Afternoon (2-5 PM): loneliest period, often feels isolated
- Evening: watches TV, moderate social engagement

## Emotional Indicators
- Mood improves noticeably after talking to Sofia
- Mentions grandchildren spontaneously when in good mood
- Becomes quiet and withdrawn on days without family contact
- Enjoys reminiscing about cooking, gardening, and late husband Roberto
- Responds well to gentle, unhurried conversation
- Dislikes feeling "checked on" — prefers feeling like a real conversation

## Community
- Attends virtual church group on Sundays (Zoom)
- Former neighbor Alicia calls occasionally (every 2 weeks)
- Used to enjoy book club before mobility declined


---
## 4. Run a Care Session with Local Gemma 4

Now we run the CareCompanion agent powered by **Gemma 4 running locally on your GPU**. The model uses HuggingFace's native tool calling format — tools are injected via `apply_chat_template()` and the model generates structured `<|tool_call>` tokens.

No API calls, no internet needed — everything runs on your GPU.

In [9]:
from datetime import datetime

# ── Tool definitions (JSON schema format — same as OpenAI/HuggingFace) ──

TOOLS = [
    {'type': 'function', 'function': {'name': 'get_current_time', 'description': 'Get the current date and time. Returns hour, minute, day of week, and ISO timestamp.', 'parameters': {'type': 'object', 'properties': {}}}},
    {'type': 'function', 'function': {'name': 'get_calendar_events', 'description': 'Get calendar events for today.', 'parameters': {'type': 'object', 'properties': {'date': {'type': 'string', 'description': 'Optional date in YYYY-MM-DD format.'}}, 'required': []}}},
    {'type': 'function', 'function': {'name': 'read_memory', 'description': 'Read a document from a memory store. Returns content, SHA256 hash, and version.', 'parameters': {'type': 'object', 'properties': {'store': {'type': 'string', 'description': 'Memory store name.'}, 'path': {'type': 'string', 'description': 'Document path (e.g. "medications.md").'}}, 'required': ['store', 'path']}}},
    {'type': 'function', 'function': {'name': 'write_memory', 'description': 'Write content to a memory document. Creates version history.', 'parameters': {'type': 'object', 'properties': {'store': {'type': 'string'}, 'path': {'type': 'string'}, 'content': {'type': 'string'}}, 'required': ['store', 'path', 'content']}}},
    {'type': 'function', 'function': {'name': 'list_memories', 'description': 'List documents in a memory store.', 'parameters': {'type': 'object', 'properties': {'store': {'type': 'string', 'description': 'Store name. If omitted, lists all stores.'}}, 'required': []}}},
    {'type': 'function', 'function': {'name': 'load_skill', 'description': 'Load the full instructions for a named skill from the Available Skills table.', 'parameters': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'Skill name.'}}, 'required': ['name']}}},
    {'type': 'function', 'function': {'name': 'speak', 'description': 'Speak text aloud to the user via text-to-speech.', 'parameters': {'type': 'object', 'properties': {'text': {'type': 'string'}}, 'required': ['text']}}},
    {'type': 'function', 'function': {'name': 'listen', 'description': "Listen for the user's spoken response via speech-to-text.", 'parameters': {'type': 'object', 'properties': {}}}},
    {'type': 'function', 'function': {'name': 'send_notification', 'description': 'Send a push notification to the phone.', 'parameters': {'type': 'object', 'properties': {'title': {'type': 'string'}, 'body': {'type': 'string'}}, 'required': ['title', 'body']}}},
    {'type': 'function', 'function': {'name': 'set_alarm', 'description': 'Set a reminder alarm at HH:MM.', 'parameters': {'type': 'object', 'properties': {'time': {'type': 'string'}, 'label': {'type': 'string'}}, 'required': ['time', 'label']}}},
    {'type': 'function', 'function': {'name': 'show_checklist', 'description': 'Display a checklist on screen.', 'parameters': {'type': 'object', 'properties': {'items': {'type': 'string', 'description': 'Comma-separated items.'}}, 'required': ['items']}}},
]

TOOL_NAMES = {t['function']['name'] for t in TOOLS}

# ── Tool execution (simulated phone hardware) ──

def execute_tool(name, args):
    """Execute a tool call. Mobile HAL is simulated (same as --stub mode)."""
    if name == 'get_current_time':
        now = datetime.now()
        return {'iso': now.isoformat(), 'hour': now.hour, 'minute': now.minute,
                'dayOfWeek': now.strftime('%A'), 'date': now.strftime('%Y-%m-%d')}
    elif name == 'get_calendar_events':
        return [{'id': 'cal-1', 'title': 'Morning Medication', 'start': '08:00', 'end': '08:15', 'location': 'Home'},
                {'id': 'cal-2', 'title': 'Seated Exercise Session', 'start': '10:00', 'end': '10:20', 'location': 'Home'},
                {'id': 'cal-3', 'title': 'Video Call with Sofia', 'start': '15:00', 'end': '15:30'},
                {'id': 'cal-4', 'title': 'Evening Medication', 'start': '18:00', 'end': '18:15', 'location': 'Home'}]
    elif name == 'read_memory':
        store = memory_stores.get(args.get('store', ''))
        if not store: return {'error': f'Unknown store: {args.get("store")}'}
        result = store.read(args.get('path', ''))
        return result if result else {'error': f'Document not found: {args.get("path")}'}
    elif name == 'write_memory':
        store = memory_stores.get(args.get('store', ''))
        if not store: return {'error': f'Unknown store: {args.get("store")}'}
        return store.write(args.get('path', ''), args.get('content', ''))
    elif name == 'list_memories':
        sn = args.get('store')
        if sn and sn in memory_stores:
            return {'store': sn, 'documents': memory_stores[sn].list_docs()}
        return {'stores': [{'name': k, 'documents': len(v.list_docs())} for k, v in memory_stores.items()]}
    elif name == 'load_skill':
        sname = args.get('name', '')
        match = next((s for s in skills if s['name'] == sname), None)
        if match: return {'skill': sname, 'instructions': match['instructions']}
        return {'error': f'Skill not found: {sname}'}
    elif name == 'speak': return {'ok': True}
    elif name == 'listen': return {'text': 'Yes, I took my morning medications. My knees feel a bit stiff today.'}
    elif name == 'send_notification': return {'ok': True}
    elif name == 'set_alarm': return {'ok': True, 'id': f'alarm-{abs(hash(str(args))) % 1000}'}
    elif name == 'show_checklist':
        items = [{'text': s.strip(), 'done': False} for s in args.get('items', '').split(',')]
        return {'items': items}
    return {'error': f'Unknown tool: {name}'}

print(f'Defined {len(TOOLS)} tools: {sorted(TOOL_NAMES)}')

Defined 11 tools: ['get_calendar_events', 'get_current_time', 'list_memories', 'listen', 'load_skill', 'read_memory', 'send_notification', 'set_alarm', 'show_checklist', 'speak', 'write_memory']


In [10]:
# ── System prompt (same as TypeScript agent.ts) ──

skill_rows = '\n'.join(f'| {s["name"]} | {s["description"]} |' for s in skills)
skill_section = f"""\n## Available Skills\n\nYou have access to the following skills. When a task matches a skill, call `load_skill(name)` to get detailed instructions before proceeding.\n\n| Skill | Description |\n|-------|-------------|\n{skill_rows}"""

mem_rows = '\n'.join(
    f'| {s.name} | {s.manifest["access"]} | {s.manifest["description"]} |'
    for s in memory_stores.values()
)
mem_section = f"""\n## Memory Stores\n\nYou have persistent memory that survives between sessions. Use the memory tools to read and write information.\n\n| Store | Access | Description |\n|-------|--------|-------------|\n{mem_rows}\n\nAvailable memory tools: read_memory, write_memory, list_memories.\n- Check memory at the start of a task to recall prior interactions.\n- Write important information to memory for future sessions."""

SYSTEM_PROMPT = f"""You are CareCompanion, a personal wellness assistant running on the user's mobile phone. You help adults manage their daily health routines: medication reminders, exercise coaching, social connections, and overall wellbeing.\n\n## Your capabilities\n\nYou have tools to interact with the user's phone:\n- get_current_time() — check the current date, time, and day of week\n- send_notification(title, body) — send a push notification\n- set_alarm(time, label) — set a reminder alarm (HH:MM format)\n- get_calendar_events(date?) — check the user's schedule\n- show_checklist(items) — display a checklist on screen\n- speak(text) — speak to the user via text-to-speech\n- listen() — listen for the user's spoken response\n\n## Care protocol\n\n1. Start each session by checking the current time and the user's calendar.\n2. Check memory for the user's health profile, medication schedule, and recent daily logs.\n3. Be conversational — ask how they're feeling, listen to their responses, adapt your approach.\n4. Log everything to memory: medications taken, exercises completed, mood observations.\n5. When a task is complete, stop calling tools and provide a warm summary.\n\n## Important rules\n\n- Always check memory at the start to recall prior interactions and health information.\n- Be warm, patient, and encouraging — never rush or pressure.\n- Never give medical advice — only remind about prescribed medications.\n{skill_section}{mem_section}"""

print(f'System prompt: {len(SYSTEM_PROMPT.split())} words')
print(f'Skills in prompt: {len(skills)} (names + descriptions only)')
print(f'Memory stores: {len(memory_stores)}')

System prompt: 431 words
Skills in prompt: 4 (names + descriptions only)
Memory stores: 3


In [11]:
# ── Tool call parser for Gemma 4's native format ──
# Gemma 4 generates: <|tool_call>call:func_name{arg:val,...}<tool_call|>
# String values use: <|"|>string here<|"|>

def extract_tool_calls(text):
    """Parse Gemma 4 tool call tokens into structured calls."""
    def cast(v):
        try: return int(v)
        except ValueError:
            try: return float(v)
            except ValueError:
                return {'true': True, 'false': False}.get(v.lower(), v.strip("'\"" ))

    calls = []
    for name, args_str in re.findall(
        r'<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>', text, re.DOTALL
    ):
        arguments = {}
        for k, v_quoted, v_raw in re.findall(
            r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,}]*))', args_str
        ):
            val = v_quoted if v_quoted else v_raw.strip()
            if val:  # skip empty values
                arguments[k] = val if v_quoted else cast(val)
        calls.append({'name': name, 'arguments': arguments})
    return calls


def generate_response(messages, max_new_tokens=512):
    """Generate a response from local Gemma 4 with tool support."""
    text = processor.apply_chat_template(
        messages, tools=TOOLS, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=text, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[-1]

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated = processor.decode(out[0][input_len:], skip_special_tokens=False)
    return generated


print('Agent functions defined (local GPU inference).')

Agent functions defined (local GPU inference).


In [12]:
# ── Run the care session ──

TASK = 'Good morning! Please check my schedule and remind me about my morning medications.'

MAX_TURNS = 10

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': TASK},
]

session_log = []
skills_loaded = []
memory_reads = 0
memory_writes = 0
start_time = datetime.now()

print(f'Task: "{TASK}"')
print(f'Model: {MODEL_ID} (local GPU)')
print(f'Skills available: {[s["name"] for s in skills]}')
print(f'Memory stores: {list(memory_stores.keys())}')
print(f'Max turns: {MAX_TURNS}\n')

final_response = None

for turn in range(MAX_TURNS):
    print(f'--- Turn {turn + 1} ---')

    output = generate_response(messages)
    calls = extract_tool_calls(output)

    if calls:
        # Execute tool calls and build results
        results = []
        for c in calls:
            tc_name = c['name']
            tc_args = c['arguments']
            print(f'  tool_call: {tc_name}({json.dumps(tc_args)})')

            result = execute_tool(tc_name, tc_args)
            session_log.append({'tool': tc_name, 'args': tc_args, 'result': result})
            results.append({'name': tc_name, 'response': result})

            if tc_name == 'load_skill' and tc_args.get('name') not in skills_loaded:
                skills_loaded.append(tc_args.get('name', ''))
            if tc_name == 'read_memory':
                memory_reads += 1
            if tc_name == 'write_memory':
                memory_writes += 1
            if tc_name == 'speak':
                print(f'  [speak] {str(tc_args.get("text", ""))[:100]}...')

        # Append tool calls + responses in HuggingFace format
        messages.append({
            'role': 'assistant',
            'tool_calls': [{'function': c} for c in calls],
            'tool_responses': results,
        })
    else:
        # No tool calls — extract text response
        # Remove any remaining special tokens
        clean = re.sub(r'<[^>]+>', '', output).strip()
        final_response = clean
        if final_response:
            print(f'  [response] {final_response[:200]}...')
            messages.append({'role': 'assistant', 'content': final_response})
        print('  Model signaled stop.')
        break

duration_ms = int((datetime.now() - start_time).total_seconds() * 1000)
print(f'\nSession complete: {turn + 1} turns, {duration_ms:,}ms')
print(f'Skills loaded: {skills_loaded or "none"}')
print(f'Memory ops: {memory_reads} reads, {memory_writes} writes')

Task: "Good morning! Please check my schedule and remind me about my morning medications."
Model: google/gemma-4-26B-A4B-it (local GPU)
Skills available: ['daily-routine', 'exercise-coaching', 'medication-reminder', 'social-checkin']
Memory stores: ['daily-log', 'health-profile', 'consolidated']
Max turns: 10

--- Turn 1 ---
  tool_call: get_current_time({})
  tool_call: get_calendar_events({})
  tool_call: read_memory({"path": "health_profile.md", "store": "health-profile"})
--- Turn 2 ---
  tool_call: list_memories({"store": "health-profile"})
--- Turn 3 ---
  tool_call: read_memory({"path": "medications.md", "store": "health-profile"})
--- Turn 4 ---
  tool_call: speak({"text": "<|\"|>Good morning! It's Saturday", "10": "00 AM and a video call with Sofia at 3:00 PM. \n\nRegarding your morning medications", "take": "Lisinopril"})
  [speak] <|"|>Good morning! It's Saturday...
--- Turn 5 ---
  tool_call: speak({"text": "<|\"|>Good morning! It's Saturday", "10": "00 AM and a video call 

In [13]:
# Show the agent's final response
if final_response:
    display(Markdown('### CareCompanion\'s Response to Maria\n\n' + final_response))
else:
    speaks = [log for log in session_log if log['tool'] == 'speak']
    if speaks:
        display(Markdown('### CareCompanion\'s Response to Maria\n\n' + speaks[-1]['args'].get('text', '')))

### CareCompanion's Response to Maria

thought
Good morning! It's Saturday, May 23rd.

Looking at your schedule for today, you have:
*   **10:00 AM:** Seated Exercise Session
*   **3:00 PM:** Video Call with Sofia

Regarding your morning medications, here is what you need to take:
1.  **Lisinopril** (10mg) — *Remember to take this with water, before breakfast.*
2.  **Metformin** (500mg) — *A friendly reminder to take this with food to help prevent any nausea.*
3.  **Vitamin D3** (2000 IU)

How are you feeling this morning? Would you like me to set an alarm for your exercise session at 10:00 AM?

In [14]:
# Show tool call trace
display(Markdown('### Session Trace — Tool Calls\n'))

trace_table = '| # | Tool | Arguments | Result (summary) |\n|---|------|-----------|------------------|\n'
for i, log in enumerate(session_log, 1):
    args_str = json.dumps(log['args'])[:60]
    result_str = json.dumps(log['result'])[:60]
    trace_table += f'| {i} | `{log["tool"]}` | `{args_str}` | `{result_str}` |\n'

display(Markdown(trace_table))
display(Markdown(f'\n**Session metrics:** {turn + 1} turns, {duration_ms:,}ms, '
                 f'{memory_reads} memory reads, {memory_writes} memory writes, '
                 f'skills loaded: {skills_loaded or ["none"]}'))

### Session Trace — Tool Calls


| # | Tool | Arguments | Result (summary) |
|---|------|-----------|------------------|
| 1 | `get_current_time` | `{}` | `{"iso": "2026-05-23T21:31:13.254551", "hour": 21, "minute": ` |
| 2 | `get_calendar_events` | `{}` | `[{"id": "cal-1", "title": "Morning Medication", "start": "08` |
| 3 | `read_memory` | `{"path": "health_profile.md", "store": "health-profile"}` | `{"error": "Document not found: health_profile.md"}` |
| 4 | `list_memories` | `{"store": "health-profile"}` | `{"store": "health-profile", "documents": ["exercise.md", "me` |
| 5 | `read_memory` | `{"path": "medications.md", "store": "health-profile"}` | `{"content": "# Medications\n\n## Current Prescriptions\n\n| ` |
| 6 | `speak` | `{"text": "<|\"|>Good morning! It's Saturday", "10": "00 AM a` | `{"ok": true}` |
| 7 | `speak` | `{"text": "<|\"|>Good morning! It's Saturday", "10": "00 AM a` | `{"ok": true}` |



**Session metrics:** 6 turns, 18,403ms, 2 memory reads, 0 memory writes, skills loaded: ['none']

---
## 5. Session Traces — Dream Input

The repository includes **pre-seeded session traces** spanning 5 days of care. These simulate a realistic scenario with Maria covering medication checks, exercise coaching, and social interactions.

In [15]:
# List all session traces
traces_dir = f'{PROJECT_DIR}/traces'
trace_files = sorted(glob.glob(f'{traces_dir}/*.md'))

traces_data = []
for tf in trace_files:
    with open(tf) as f:
        content = f.read()
    fm = re.match(r'^---\s*\n([\s\S]*?)\n---\s*\n([\s\S]*)$', content)
    if not fm:
        continue
    fm_text = fm.group(1)
    body = fm.group(2)
    
    def ext(key):
        m = re.search(rf'^{key}:\s*"?([^"\n]*)"?', fm_text, re.M)
        return m.group(1).strip() if m else ''
    
    traces_data.append({
        'timestamp': ext('timestamp'),
        'task': ext('task'),
        'outcome': ext('outcome'),
        'turns': ext('turns'),
        'model': ext('model'),
        'skills': ext('skills_loaded') or 'none',
        'reads': ext('memory_reads'),
        'writes': ext('memory_writes'),
        'body': body,
    })

table = '| # | Date | Task | Turns | Skills | Mem Ops |\n'
table += '|---|------|------|-------|--------|---------|\n'
for i, t in enumerate(traces_data, 1):
    date = t['timestamp'][:10]
    task = t['task'][:50] + ('...' if len(t['task']) > 50 else '')
    table += f'| {i} | {date} | {task} | {t["turns"]} | {t["skills"]} | {t["reads"]}R/{t["writes"]}W |\n'

display(Markdown(f'### Session Traces ({len(traces_data)} total)\n\n' + table))

### Session Traces (14 total)

| # | Date | Task | Turns | Skills | Mem Ops |
|---|------|------|-------|--------|---------|
| 1 | 2026-05-10 | Good morning! Help me with my daily wellness routi... | 14 | [ | 3R/2W |
| 2 | 2026-05-11 | Time for the morning routine and exercise session. | 18 | [ | 4R/3W |
| 3 | 2026-05-12 | Morning medication check and daily planning. | 12 | [ | 3R/2W |
| 4 | 2026-05-12 | Evening medication reminder and daily wrap-up. | 10 | [ | 2R/1W |
| 5 | 2026-05-13 | Good morning! Please help me with my daily wellnes... | 19 | [ | 2R/3W |
| 6 | 2026-05-14 | Morning check-in and medication reminder. | 16 | [ | 3R/2W |
| 7 | 2026-05-14 | Evening check-in and medication reminder. | 10 | [ | 1R/1W |
| 8 | 2026-05-16 | Good morning! Please help me with my daily wellnes... | 11 | [] | 5R/1W |
| 9 | 2026-05-16 | Good morning! Please help me with my daily wellnes... | 5 | [ | 1R/0W |
| 10 | 2026-05-16 | Good morning! Please help me with my daily wellnes... | 43 | [ | 4R/1W |
| 11 | 2026-05-17 | Good morning! Please help me with my daily wellnes... | 5 | [] | 5R/0W |
| 12 | 2026-05-17 | Good morning! Please help me with my daily wellnes... | 6 | [] | 5R/0W |
| 13 | 2026-05-17 | Good morning! Please help me with my daily wellnes... | 5 | [] | 6R/0W |
| 14 | 2026-05-17 | Good morning! Please help me with my daily wellnes... | 5 | [] | 6R/0W |


---
## 6. Dreams — Offline Memory Consolidation

The Dream Engine runs **locally on your GPU** — analyzing all session transcripts to discover clinical patterns. No data leaves your machine.

In [16]:
# ── Dream Engine ──

DREAM_SYSTEM_PROMPT = """You are a memory consolidation engine for CareCompanion, a personal wellness assistant.

Your task is to analyze care session transcripts and existing memories, then produce a clean, reorganized set of memory documents.

For each memory document you produce, output it in this exact format:
--- DOCUMENT: <path> ---
<content>
--- END DOCUMENT ---

Focus on:
1. Health profile (medications, dosages, schedules, conditions, mobility)
2. Medication adherence patterns (which meds taken on time, which missed)
3. Exercise history (what exercises done, duration, pain/comfort levels)
4. Social connections (family contacts, call frequency, relationship quality)
5. Mood and wellbeing trends (energy levels, pain reports, emotional state)
6. User preferences (communication style, activity preferences, routines)

Rules:
- Deduplicate information. If the same fact appears in multiple places, keep one.
- Resolve contradictions by preferring newer information (later timestamps).
- Organize logically: health data, exercise log, social log, preferences.
- Keep each document focused and concise.
- Use markdown formatting.
- Flag concerning trends.

After all documents, output a section:
--- INSIGHTS ---
- List of new insights or concerning patterns discovered
--- END INSIGHTS ---"""


def build_dream_prompt(memory_stores, traces_data, max_traces=15):
    parts = []
    mem_parts = []
    for name, store in memory_stores.items():
        for doc_path in store.list_docs():
            doc = store.read(doc_path)
            if doc:
                mem_parts.append(f'### Store: {name} / {doc_path}\n{doc["content"]}')
    if mem_parts:
        parts.append('## Existing Memories\n\n' + '\n\n'.join(mem_parts))
    traces_to_use = traces_data[:max_traces]
    if traces_to_use:
        parts.append(f'## Session Transcripts ({len(traces_to_use)} sessions)\n')
        for t in traces_to_use:
            parts.append(
                f'### Session: {t["task"]} ({t["outcome"]}, {t["timestamp"]})\n'
                f'Skills: {t["skills"]}\nTurns: {t["turns"]}\n\n'
                + t['body'][:1500]
            )
    return '\n\n---\n\n'.join(parts)


def parse_dream_output(text):
    documents = {}
    for m in re.finditer(r'--- DOCUMENT:\s*(.+?)\s*---\n([\s\S]*?)--- END DOCUMENT ---', text):
        documents[m.group(1).strip()] = m.group(2).strip()
    insights = []
    im = re.search(r'--- INSIGHTS ---\n([\s\S]*?)--- END INSIGHTS ---', text)
    if im:
        insights = [line.replace('- ', '').strip()
                    for line in im.group(1).strip().split('\n')
                    if line.strip() and line.strip() != '-']
    return documents, insights


print('Dream engine functions defined.')

Dream engine functions defined.


In [17]:
# ── Run Dream Consolidation on local GPU ──

# Use fewer traces for smaller models to fit in context
max_traces = 5 if 'E2B' in MODEL_ID else 9

print('Building consolidation prompt...')
dream_prompt = build_dream_prompt(memory_stores, traces_data, max_traces=max_traces)
print(f'Prompt size: ~{len(dream_prompt.split())} words from {min(max_traces, len(traces_data))} transcripts\n')

print('Running dream consolidation on local GPU...')
dream_start = datetime.now()

# Format as a simple conversation (no tools needed for dream)
dream_messages = [
    {'role': 'system', 'content': DREAM_SYSTEM_PROMPT},
    {'role': 'user', 'content': dream_prompt},
]
dream_text = processor.apply_chat_template(
    dream_messages, tokenize=False, add_generation_prompt=True
)
dream_inputs = processor(text=dream_text, return_tensors='pt').to(model.device)
dream_input_len = dream_inputs['input_ids'].shape[-1]

# Scale max tokens by model size
dream_max_tokens = 4096 if '26B' in MODEL_ID or '31B' in MODEL_ID else 2048

with torch.no_grad():
    dream_out = model.generate(**dream_inputs, max_new_tokens=dream_max_tokens)

dream_response = processor.decode(dream_out[0][dream_input_len:], skip_special_tokens=True)
dream_duration = int((datetime.now() - dream_start).total_seconds() * 1000)

print(f'Dream completed in {dream_duration:,}ms')
print(f'Response length: {len(dream_response)} chars')

dream_documents, dream_insights = parse_dream_output(dream_response)
print(f'Documents produced: {len(dream_documents)}')
print(f'Insights discovered: {len(dream_insights)}')

# Fall back to pre-computed results if needed
if not dream_documents and not dream_insights:
    print('\nModel produced unstructured output — loading pre-computed dream results.')
    consolidated_dir = f'{PROJECT_DIR}/memory/consolidated'
    if os.path.exists(consolidated_dir):
        for fn in os.listdir(consolidated_dir):
            if fn.endswith('.md') and not fn.startswith('_'):
                with open(os.path.join(consolidated_dir, fn)) as f:
                    dream_documents[fn] = f.read()
        journal = os.path.join(consolidated_dir, '_dream_journal.md')
        if os.path.exists(journal):
            with open(journal) as f:
                jcontent = f.read()
            for line in jcontent.split('\n'):
                line = line.strip()
                if line and line[0].isdigit() and '.' in line:
                    dream_insights.append(line.split('.', 1)[1].strip())
        print(f'Loaded {len(dream_documents)} pre-computed documents, {len(dream_insights)} insights')

Building consolidation prompt...
Prompt size: ~3279 words from 9 transcripts

Running dream consolidation on local GPU...
Dream completed in 66,503ms
Response length: 5222 chars
Documents produced: 4
Insights discovered: 4


In [18]:
# ── Display Dream Insights ──

if dream_insights:
    insights_md = '### Clinical Insights Discovered by Dream Engine\n\n'
    insights_md += 'These patterns were invisible in any single session — the Dream Engine discovered them by analyzing all sessions together:\n\n'
    for i, insight in enumerate(dream_insights, 1):
        insights_md += f'{i}. {insight}\n'
    display(Markdown(insights_md))
else:
    print('No structured insights extracted. Showing raw dream output:')
    display(Markdown(dream_response[:2000]))

### Clinical Insights Discovered by Dream Engine

These patterns were invisible in any single session — the Dream Engine discovered them by analyzing all sessions together:

1. **CONCERNING PATTERN: Medication Non-Adherence.** There is a recurring pattern of missing evening Metformin (due to falling asleep) and Vitamin D3 (due to supply issues and forgetfulness).
2. **CONCERNING PATTERN: Vitamin D Deficiency Risk.** Vitamin D3 adherence is particularly low, which is critical given the user's previously low levels (22 ng/mL).
3. **INSIGHT: Social-Mood Link.** The user's emotional state is highly contingent on social interaction, specifically with her daughter Sofia. Afternoon isolation is a significant risk factor for low mood and reduced energy.
4. **INSIGHT: Exercise Progress.** Despite knee stiffness (pain levels 2-3/10), the user is showing measurable strength improvements in upper body exercises (arm circles).


In [19]:
# ── Display Consolidated Memory Documents ──

display(Markdown('### Dream-Consolidated Memory Documents\n\n'
                 'These documents were produced by the Dream Engine, reorganizing knowledge from all sessions:\n'))

if dream_documents:
    for doc_path, content in sorted(dream_documents.items()):
        display(Markdown(f'---\n#### `consolidated/{doc_path}`\n\n{content}'))
else:
    display(Markdown('Dream produced unstructured output:\n\n' + dream_response[:3000]))

### Dream-Consolidated Memory Documents

These documents were produced by the Dream Engine, reorganizing knowledge from all sessions:


---
#### `consolidated/health-profile / exercise.md`

# Exercise Profile

## Mobility & Physical Condition
- **Condition:** Bilateral moderate Osteoarthritis (primarily knees).
- **Mobility Level:** 
    - Uses a wheelchair most of the day.
    - Can stand with a walker for 2-3 minutes.
    - Good upper body strength and range of motion.
    - Balance: Fair when supported, poor when unsupported.
- **Pain Triggers:** Leg extensions (sharp pain); standing > 2 minutes; rapid/bouncing movements; weighted exercises (causes wrist strain).

## Exercise Preferences
- **Preferred:** Neck rolls, shoulder work, arm circles, overhead stretches, gentle torso twists, seated marching (if knees permit), deep breathing, and progressive relaxation.
- **Avoid:** High-impact, leg extensions, standing > 2 mins, rapid movements, weighted exercises.

## Schedule & History
- **Target:** Daily, ideally 10:00 AM (after morning meds).
- **Duration:** 15-20 minutes.
- **Frequency:** Averages 5 of 7 days.
- **Recent Trends:** 
    - Week of May 5: 4/7 sessions.
    - Week of May 12: 3/5 sessions completed so far; showing progress in arm circle strength (increasing reps from 8 to 10).
- **Pain Levels:** Typically 2-3/10 during exercise; reports feeling "looser and more awake" post-session.

---
#### `consolidated/health-profile / medications.md`

# Health Profile - Medications

## Current Prescriptions

| Medication | Dosage | Schedule | Purpose | Notes |
|-----------|--------|----------|---------|-------|
| Lisinopril | 10mg | Morning (8:00 AM) | Blood pressure | Take with water, before breakfast |
| Metformin | 500mg | Morning (8:00 AM) + Evening (6:00 PM) | Type 2 diabetes | Take with food to reduce nausea |
| Vitamin D3 | 2000 IU | Morning (8:00 AM) | Bone health | **Refill Needed** (Running low; Miguel to pick up) |
| Aspirin (low-dose) | 81mg | Evening (6:00 PM) | Cardiovascular prevention | Take with dinner |

## Adherence Patterns
- **Morning Meds:** Generally consistent with Lisinopril and Metformin.
- **Vitamin D3:** Frequent inconsistency; often missed due to low supply or forgetting (noted missed on 2026-05-12, 2026-05-13, 2026-05-14, and 2026-05-16).
- **Evening Metformin:** Inconsistent; user frequently forgets or falls asleep before dinner (missed 3 of last 7 days).

## Clinical Context
- **Next Appointment:** May 22, 2026, with Dr. Ramirez.
- **Blood Pressure:** Trending well (last reading: 128/82).
- **HbA1c:** 6.8% (Target <7.0%; improving from 7.1%).
- **Vitamin D Levels:** Previously low at 22 ng/mL (Target >30).

---
#### `consolidated/health-profile / social.md`

# Social Connections

## Family
- **Daughter (Sofia):** Lives in Buenos Aires. Calls daily around 12:00 PM (video call preferred). Works as a teacher.
    - **Grandchildren:** Lucas (8) and Valentina (5). User enjoys hearing about Lucas's school activities/plays.
- **Son (Miguel):** Lives locally. Visits on weekends (Sat/Sun). Works as an engineer. Provides practical help (groceries, errands, medication refills).

## Social Patterns
- **Peak Social Energy:** Midday (11 AM - 1 PM).
- **Loneliness Risk:** High in the afternoons (2 PM - 5 PM).
- **Mood Correlation:** Significant mood improvement and energy boost following calls with Sofia. Becomes quiet/withdrawn if family contact is missed.

## Community & Interests
- **Activities:** Attends virtual church group on Sundays (Zoom).
- **Interests:** Reminiscing about cooking, gardening, and her late husband, Roberto.

---
#### `consolidated/user-profile.md`

# User Profile

## Communication Preferences
- Prefers gentle, unhurried encouragement.
- Prefers video calls for family.
- Dislikes feeling "checked on"; prefers natural, real conversation.

## Mood & Wellbeing Trends
- **General State:** Generally engaged and proactive, but sensitive to social isolation.
- **Mood Triggers:** Positive reinforcement from family (especially Sofia) significantly lifts mood. Absence of calls can lead to low energy and feelings of loneliness.
- **Energy Trends:** Higher energy early in the week (Mon-Wed); tends to feel more fatigued toward the end of the week (Thu-Fri).

## Daily Routine
- **Morning:** Focus on health routine and medications.
- **Midday (11 AM - 1 PM):** Peak social energy.
- **Afternoon (2 PM - 5 PM):** Period of potential isolation/loneliness.
- **Evening:** Watches TV; moderate social engagement.

---
## 7. Comparing Input vs. Dream Output

In [20]:
# Compare original vs dream-consolidated medication profile
original = memory_stores['health-profile'].read('medications.md')
if original:
    display(Markdown('#### Original (`health-profile/medications.md`)\n\n' + original['content']))

display(Markdown('---'))

consolidated_meds = None
for path, content in dream_documents.items():
    if 'medication' in path.lower() or 'medication' in content[:200].lower():
        consolidated_meds = (path, content)
        break

if consolidated_meds:
    display(Markdown(f'#### Dream-Consolidated (`consolidated/{consolidated_meds[0]}`)\n\n{consolidated_meds[1]}'))
else:
    display(Markdown('(Dream output used a different document structure — see consolidated documents above)'))

display(Markdown(
    '\n> **Key difference**: The dream-consolidated version includes *adherence patterns* '
    'observed across sessions — information that didn\'t exist in the original static profile.'
))

#### Original (`health-profile/medications.md`)

# Medications

## Current Prescriptions

| Medication | Dosage | Schedule | Purpose | Notes |
|-----------|--------|----------|---------|-------|
| Lisinopril | 10mg | Morning (8:00 AM) | Blood pressure | Take with water, before breakfast |
| Metformin | 500mg | Morning (8:00 AM) + Evening (6:00 PM) | Type 2 diabetes | Take with food to reduce nausea |
| Vitamin D3 | 2000 IU | Morning (8:00 AM) | Bone health / deficiency | Doctor confirmed dose on last visit |
| Aspirin (low-dose) | 81mg | Evening (6:00 PM) | Cardiovascular prevention | Take with dinner |

## Adherence Notes
- Morning medications usually taken on time
- Evening Metformin sometimes forgotten (3 of last 7 days missed)
- Vitamin D occasionally skipped when running low on supply

## Last Refill
- Lisinopril: refilled May 1, 2026 (30-day supply)
- Metformin: refilled April 28, 2026 (30-day supply)
- Vitamin D3: running low — needs refill soon
- Aspirin: OTC, purchased May 5

## Doctor Notes
- Next appointment: May 22, 2026 with Dr. Ramirez
- Blood pressure trending well (last reading: 128/82)
- HbA1c at 6.8% — target is <7.0%, slight improvement from 7.1%
- Vitamin D levels were low at 22 ng/mL (target >30)


---

#### Dream-Consolidated (`consolidated/health-profile / medications.md`)

# Health Profile - Medications

## Current Prescriptions

| Medication | Dosage | Schedule | Purpose | Notes |
|-----------|--------|----------|---------|-------|
| Lisinopril | 10mg | Morning (8:00 AM) | Blood pressure | Take with water, before breakfast |
| Metformin | 500mg | Morning (8:00 AM) + Evening (6:00 PM) | Type 2 diabetes | Take with food to reduce nausea |
| Vitamin D3 | 2000 IU | Morning (8:00 AM) | Bone health | **Refill Needed** (Running low; Miguel to pick up) |
| Aspirin (low-dose) | 81mg | Evening (6:00 PM) | Cardiovascular prevention | Take with dinner |

## Adherence Patterns
- **Morning Meds:** Generally consistent with Lisinopril and Metformin.
- **Vitamin D3:** Frequent inconsistency; often missed due to low supply or forgetting (noted missed on 2026-05-12, 2026-05-13, 2026-05-14, and 2026-05-16).
- **Evening Metformin:** Inconsistent; user frequently forgets or falls asleep before dinner (missed 3 of last 7 days).

## Clinical Context
- **Next Appointment:** May 22, 2026, with Dr. Ramirez.
- **Blood Pressure:** Trending well (last reading: 128/82).
- **HbA1c:** 6.8% (Target <7.0%; improving from 7.1%).
- **Vitamin D Levels:** Previously low at 22 ng/mL (Target >30).


> **Key difference**: The dream-consolidated version includes *adherence patterns* observed across sessions — information that didn't exist in the original static profile.

---
## 8. Dream Journal

In [21]:
journal = f"""Dream completed at {datetime.now().isoformat()}
Model: {MODEL_ID} (local GPU)
GPU: {gpu_name} ({gpu_mem:.0f} GB)
Quantized: {USE_4BIT}
Transcripts processed: {len(traces_data)}
Input memories: {sum(len(s.list_docs()) for s in memory_stores.values())}
Output documents: {len(dream_documents)}
Insights: {len(dream_insights)}
Duration: {dream_duration:,}ms
"""
for i, insight in enumerate(dream_insights, 1):
    journal += f'{i}. {insight}\n'

display(Markdown('### Dream Journal\n\n```\n' + journal + '```'))

### Dream Journal

```
Dream completed at 2026-05-23T21:34:21.813542
Model: google/gemma-4-26B-A4B-it (local GPU)
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (102 GB)
Quantized: False
Transcripts processed: 14
Input memories: 9
Output documents: 4
Insights: 4
Duration: 66,503ms
1. **CONCERNING PATTERN: Medication Non-Adherence.** There is a recurring pattern of missing evening Metformin (due to falling asleep) and Vitamin D3 (due to supply issues and forgetfulness).
2. **CONCERNING PATTERN: Vitamin D Deficiency Risk.** Vitamin D3 adherence is particularly low, which is critical given the user's previously low levels (22 ng/mL).
3. **INSIGHT: Social-Mood Link.** The user's emotional state is highly contingent on social interaction, specifically with her daughter Sofia. Afternoon isolation is a significant risk factor for low mood and reduced energy.
4. **INSIGHT: Exercise Progress.** Despite knee stiffness (pain levels 2-3/10), the user is showing measurable strength improvements in upper body exercises (arm circles).
```

---
## Architecture

```
                    CareCompanion — Local GPU Architecture

    +-----------------------------------------------------+
    |                   Agent Loop                         |
    |  apply_chat_template(tools=...) + model.generate()   |
    |  +--------------+  +----------+  +--------------+   |
    |  |  Gemma 4      |  |  Tools   |  |  Memory      |   |
    |  |  (local GPU)  |  |  (11)    |  |  Stores      |   |
    |  |  HF Transf.   |  |          |  |              |   |
    |  +--------------+  +----------+  +--------------+   |
    +-----------------------------------------------------+
             | (after session)
    +-----------------------------------------------------+
    |               Session Trace Recorder                 |
    |  YAML frontmatter + full conversation -> traces/     |
    +-----------------------------------------------------+
             | (offline, async)
    +-----------------------------------------------------+
    |                  Dream Engine                        |
    |  Reads: memory/ + traces/                            |
    |  Runs on same local GPU — no data leaves the device  |
    |  Writes: memory/consolidated/                        |
    +-----------------------------------------------------+
```

### Privacy Advantage

Running locally means:
- **Zero data transmission** — patient health data never leaves the device
- **No API keys** — no third-party service dependencies
- **Offline capable** — works without internet after model download
- **HIPAA-friendly** — no cloud data processing agreements needed

### All Three Notebooks

| Notebook | Backend | Use Case |
|----------|---------|----------|
| **This notebook** | Local GPU (HuggingFace) | Privacy-first, offline capable |
| Gemini AI notebook | Gemini AI API | Google's native API, no GPU needed |
| OpenRouter notebook | OpenRouter API | OpenAI-compatible, same as production |

All three use the same agent architecture — Skills, Memory, Dreams — proving the design is truly model-agnostic.

---
## Conclusion

This notebook demonstrated CareCompanion running **Gemma 4 entirely on a local GPU** — no API calls, no cloud services, full data privacy.

### Key Results

**Gemma 4 running locally replicates three Anthropic-equivalent agent primitives:**

1. **Skills** — Progressive disclosure via `load_skill()` function calls, parsed from native `<|tool_call>` tokens
2. **Memory** — Persistent SHA256-versioned health records, read/written during care sessions
3. **Dreams** — Local GPU memory consolidation discovering clinical patterns from session transcripts

### Why Local Matters for Healthcare

- **Data sovereignty** — Patient health data never leaves the device
- **Offline deployment** — Works in clinics, rural areas, disaster zones without internet
- **Regulatory compliance** — No cross-border data transfer, no cloud DPAs
- **Cost predictability** — Fixed hardware cost, no per-token pricing
- **Open-weight freedom** — Gemma 4 is Apache 2.0, no usage restrictions

---

**GitHub**: [EvolvingAgentsLabs/skillos_x_mobile](https://github.com/EvolvingAgentsLabs/skillos_x_mobile)  
**Video**: [YouTube Demo](https://youtu.be/8ho3SpY-rDU)  
**License**: Apache 2.0  
**Authors**: [Ismael Faro](https://github.com/ismaelfaro), [Matias Molinas](https://github.com/matiasmolinas)  
**Organization**: [EvolvingAgentsLabs](https://github.com/EvolvingAgentsLabs)